In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("data/clean/train.csv")
test = pd.read_csv("data/clean/test.csv")
features = [c for c in train.columns if c != "default_2yrs"]
features

['utilisation',
 'age',
 'late_30_59',
 'debt_ratio',
 'monthly_income',
 'open_credit_lines',
 'late_90',
 'real_estate_loans',
 'late_60_89',
 'dependents']

In [2]:
inf = np.inf
bins = {
    "utilisation":       [-inf, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, inf],
    "age":               [-inf, 30, 40, 50, 60, 70, inf],
    "late_30_59":        [-inf, 0, 1, 2, inf],
    "late_60_89":        [-inf, 0, 1, inf],
    "late_90":           [-inf, 0, 1, inf],
    "debt_ratio":        [-inf, 0.2, 0.4, 0.6, 0.8, 1.0, inf],
    "monthly_income":    [-inf, 2000, 3500, 5000, 7500, 10000, inf],
    "open_credit_lines": [-inf, 2, 5, 10, 15, inf],
    "real_estate_loans": [-inf, 0, 1, 2, inf],
    "dependents":        [-inf, 0, 1, 2, inf],
}

In [3]:
def make_bins(data, col):
    b = pd.cut(data[col], bins[col])
    return b.cat.add_categories("Missing").fillna("Missing")

def woe_table(data, col):
    b = make_bins(data, col)
    t = data.groupby(b, observed=False)["default_2yrs"].agg(total="size", bad="sum")
    t["good"] = t["total"] - t["bad"]
    t = t[t["total"] > 0]
    t["bad_rate"] = (t["bad"] / t["total"]).round(4)
    pct_good = (t["good"] + 0.5) / t["good"].sum()
    pct_bad = (t["bad"] + 0.5) / t["bad"].sum()
    t["woe"] = np.log(pct_good / pct_bad).round(4)
    t["iv"] = ((pct_good - pct_bad) * t["woe"]).round(4)
    t.index = t.index.astype(str)
    return t

woe_table(train, "utilisation")

,total,bad,good,bad_rate,woe,iv
utilisation,,,,,,
"(-inf, 0.1]",45152,835,44317,0.0185,1.3348,0.4448
"(0.1, 0.3]",19731,587,19144,0.0298,0.8476,0.0947
"(0.3, 0.5]",11187,648,10539,0.0579,0.1519,0.0023
"(0.5, 0.7]",7919,784,7135,0.0990,-0.4285,0.0167
"(0.7, 0.9]",7005,1043,5962,0.1489,-0.8934,0.0785
"(0.9, 1.0]",11675,2260,9415,0.1936,-1.2095,0.2734
"(1.0, inf]",2157,845,1312,0.3917,-2.1965,0.2352
Missing,173,16,157,0.0925,-0.3802,0.0003


In [4]:
iv = pd.Series({col: woe_table(train, col)["iv"].sum() for col in features}).sort_values(ascending=False).round(3)

def strength(v):
    if v < 0.02: return "Useless"
    if v < 0.1:  return "Weak"
    if v < 0.3:  return "Medium"
    if v < 0.5:  return "Strong"
    return "Very strong (check!)"

iv_table = pd.DataFrame({"IV": iv, "strength": iv.apply(strength)})
iv_table

,IV,strength
utilisation,1.146,Very strong (check!)
late_90,0.887,Very strong (check!)
late_30_59,0.739,Very strong (check!)
late_60_89,0.600,Very strong (check!)
age,0.226,Medium
open_credit_lines,0.079,Weak
monthly_income,0.074,Weak
debt_ratio,0.064,Weak
real_estate_loans,0.056,Weak
dependents,0.036,Weak


In [5]:
def to_woe(data):
    out = pd.DataFrame(index=data.index)
    for col in features:
        t = woe_table(train, col)
        out[col] = make_bins(data, col).astype(str).map(t["woe"])
    return out

X_train, X_test = to_woe(train), to_woe(test)
y_train, y_test = train["default_2yrs"], test["default_2yrs"]
print("Blanks after mapping:", X_train.isna().sum().sum(), X_test.isna().sum().sum())

Blanks after mapping: 0 0


In [6]:
X_train.corr().round(2)

,utilisation,age,late_30_59,debt_ratio,monthly_income,open_credit_lines,late_90,real_estate_loans,late_60_89,dependents
utilisation,1.00,0.27,0.27,0.14,0.11,0.22,0.29,0.09,0.23,0.10
age,0.27,1.00,0.10,0.09,0.09,0.07,0.10,0.01,0.08,0.27
late_30_59,0.27,0.10,1.00,0.08,0.03,-0.01,0.28,0.00,0.32,0.07
debt_ratio,0.14,0.09,0.08,1.00,0.31,-0.12,-0.00,-0.14,0.03,0.14
monthly_income,0.11,0.09,0.03,0.31,1.00,0.10,0.08,0.13,0.05,-0.05
open_credit_lines,0.22,0.07,-0.01,-0.12,0.10,1.00,0.16,0.28,0.06,-0.11
late_90,0.29,0.10,0.28,-0.00,0.08,0.16,1.00,0.08,0.35,0.03
real_estate_loans,0.09,0.01,0.00,-0.14,0.13,0.28,0.08,1.00,0.04,-0.12
late_60_89,0.23,0.08,0.32,0.03,0.05,0.06,0.35,0.04,1.00,0.03
dependents,0.10,0.27,0.07,0.14,-0.05,-0.11,0.03,-0.12,0.03,1.00


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pd.Series(model.coef_[0], index=features).round(3).sort_values()

debt_ratio          -0.837
utilisation         -0.616
real_estate_loans   -0.578
late_90             -0.532
late_30_59          -0.491
late_60_89          -0.375
age                 -0.359
dependents          -0.295
monthly_income      -0.079
open_credit_lines   -0.038
dtype: float64

In [8]:
from sklearn.metrics import roc_auc_score, roc_curve

p_train = model.predict_proba(X_train)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]

gini_train = 2 * roc_auc_score(y_train, p_train) - 1
gini_test = 2 * roc_auc_score(y_test, p_test) - 1
fpr, tpr, _ = roc_curve(y_test, p_test)
ks = (tpr - fpr).max()

print(f"Gini train: {gini_train:.3f}")
print(f"Gini test:  {gini_test:.3f}")
print(f"KS test:    {ks:.3f}")

Gini train: 0.715
Gini test:  0.716
KS test:    0.563


In [9]:
pdo, base_score, base_odds = 20, 600, 50
factor = pdo / np.log(2)
offset = base_score - factor * np.log(base_odds)
n = len(features)
b0 = model.intercept_[0]

rows = []
for col, b in zip(features, model.coef_[0]):
    t = woe_table(train, col)
    for bin_, w in t["woe"].items():
        pts = -factor * (b * w + b0 / n) + offset / n
        rows.append({"feature": col, "bin": bin_, "woe": w, "points": round(pts)})

scorecard = pd.DataFrame(rows)
scorecard[scorecard["feature"] == "utilisation"]

,feature,bin,woe,points
0,utilisation,"(-inf, 0.1]",1.3348,80
1,utilisation,"(0.1, 0.3]",0.8476,71
2,utilisation,"(0.3, 0.5]",0.1519,59
3,utilisation,"(0.5, 0.7]",-0.4285,49
4,utilisation,"(0.7, 0.9]",-0.8934,40
5,utilisation,"(0.9, 1.0]",-1.2095,35
6,utilisation,"(1.0, inf]",-2.1965,17
7,utilisation,Missing,-0.3802,49


In [10]:
def score(data):
    s = pd.Series(0.0, index=data.index)
    for col in features:
        pts = scorecard[scorecard["feature"] == col].set_index("bin")["points"]
        s += make_bins(data, col).astype(str).map(pts)
    return s

train["score"] = score(train)
test["score"] = score(test)

print(test["score"].describe().round(0))
print("Gini using points:", round(2 * roc_auc_score(y_test, -test["score"]) - 1, 3))

count    45000.0
mean       581.0
std         34.0
min        403.0
25%        565.0
50%        591.0
75%        606.0
max        628.0
Name: score, dtype: float64
Gini using points: 0.715


In [11]:
test["decile"] = pd.qcut(test["score"], 10, labels=False, duplicates="drop") + 1
bands = test.groupby("decile").agg(
    min_score=("score", "min"), max_score=("score", "max"),
    customers=("score", "size"), bad_rate=("default_2yrs", "mean"))
bands["bad_rate"] = (bands["bad_rate"] * 100).round(1)
bands

,min_score,max_score,customers,bad_rate
decile,,,,
1,403.0,538.0,4533,36.4
2,539.0,558.0,4697,11.2
3,559.0,571.0,4589,6.1
4,572.0,582.0,4444,4.1
5,583.0,591.0,4747,2.7
6,592.0,597.0,4041,2.2
7,598.0,603.0,4881,1.2
8,604.0,609.0,5016,1.0
9,610.0,613.0,3704,0.5


In [12]:
import os
os.makedirs("outputs", exist_ok=True)
scorecard.to_csv("outputs/scorecard_points.csv", index=False)
test.to_csv("data/clean/test_scored.csv", index=False)